# ML Fundamentals #4: Random Forest

## Crop Recommendation using Random Forest

This project implements a Random Forest Classifier to recommend the most suitable crop based on soil nutrients and environmental conditions.

The notebook covers:

- Data Loading
- Dataset Understanding
- Exploratory Data Analysis
- Data Preprocessing
- Random Forest Training
- Out-of-Bag Evaluation
- Prediction
- Model Evaluation
- Feature Importance
- Model Serialization

In [1]:
# Import required libraries for data analysis, visualization, modeling, and saving outputs

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [2]:
# Create output directories for storing figures and tables

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

print("Output directories created successfully.")

Output directories created successfully.


## 2. Loading the Dataset

This project uses the **Crop Recommendation Dataset**, which contains soil nutrient measurements and environmental conditions to recommend the most suitable crop for cultivation.

The dataset is a multi-class classification problem where each record represents one agricultural observation.

In [5]:
# Load the Crop Recommendation dataset

df = pd.read_csv("data/raw/Crop_recommendation.csv")

df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [6]:
# Save a preview of the dataset

df.head(10).to_csv(
    "outputs/tables/dataset_preview.csv",
    index=False
)

In [7]:
# Display dataset dimensions

print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

Rows : 2200
Columns : 8


### Insights

- The Crop Recommendation dataset has been successfully loaded.
- Each row represents one agricultural observation.
- The dataset contains soil nutrient values and environmental measurements.
- The target variable represents the recommended crop.

## 3. Understanding the Dataset

Before training the Random Forest model, we examine the dataset structure, feature information, missing values, statistical summary, and crop class distribution.

In [8]:
# Display dataset information

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   str    
dtypes: float64(4), int64(3), str(1)
memory usage: 137.6 KB


In [9]:
# Generate descriptive statistics

summary = df.describe().T

summary

,count,mean,std,min,25%,50%,75%,max
N,2200.0,50.551818,36.917334,0.000000,21.000000,37.000000,84.250000,140.000000
P,2200.0,53.362727,32.985883,5.000000,28.000000,51.000000,68.000000,145.000000
K,2200.0,48.149091,50.647931,5.000000,20.000000,32.000000,49.000000,205.000000
temperature,2200.0,25.616244,5.063749,8.825675,22.769375,25.598693,28.561654,43.675493
humidity,2200.0,71.481779,22.263812,14.258040,60.261953,80.473146,89.948771,99.981876
ph,2200.0,6.469480,0.773938,3.504752,5.971693,6.425045,6.923643,9.935091
rainfall,2200.0,103.463655,54.958389,20.211267,64.551686,94.867624,124.267508,298.560117


In [10]:
# Save descriptive statistics

summary.to_csv(
    "outputs/tables/dataset_summary.csv"
)

In [11]:
# Check for missing values

missing = df.isnull().sum()

missing

N              0
P              0
K              0
temperature    0
humidity       0
ph             0
rainfall       0
label          0
dtype: int64

In [12]:
# Save missing value summary

missing.to_csv(
    "outputs/tables/missing_values.csv"
)

In [13]:
# Display crop class distribution

df["label"].value_counts()

label
rice           100
maize          100
chickpea       100
kidneybeans    100
pigeonpeas     100
mothbeans      100
mungbean       100
blackgram      100
lentil         100
pomegranate    100
banana         100
mango          100
grapes         100
watermelon     100
muskmelon      100
apple          100
orange         100
papaya         100
coconut        100
cotton         100
jute           100
coffee         100
Name: count, dtype: int64

In [14]:
# Save crop class distribution

crop_distribution = df["label"].value_counts().reset_index()

crop_distribution.columns = ["Crop", "Count"]

crop_distribution.to_csv(
    "outputs/tables/crop_distribution.csv",
    index=False
)

### Insights

- The dataset contains numerical soil and environmental features.
- The target variable is the recommended crop label.
- No missing values are present.
- The dataset contains multiple crop classes, making this a multi-class classification problem.